# 14 — Property, Table-Driven, and Integration Testing

Use the numbered concept checks where you need practice. For the engineering route, complete **Catch a stateful regression** and the independent build.

In [ ]:
# Setup: run this first. On Google Colab (or anywhere the course files are missing) it
# downloads the course and moves into this notebook's folder. Elsewhere it does nothing.
import os, sys
if not os.path.exists("../README.md"):
    course = os.path.join(os.path.expanduser("~"), "python-notebook")
    if not os.path.isdir(course):
        os.system(f"git clone -q --depth 1 https://github.com/SwarmLadder/python-notebook {course}")
    os.chdir(os.path.join(course, "advanced", "notebooks"))
    sys.path.insert(0, os.getcwd())

In [ ]:
from lesson import Exercises, ___, run

check = Exercises('advanced/14', {1: '3fd3b89960bbd2a8', 2: '34ff1a6cdd7cb5c5', 3: '7664c53c99c92e3b', 4: 'a380fb220faa183b', 5: '685eb3d4fa9c94df', 6: 'bd7c48a70e0f95dc', 7: '68958d238e75f3f4'}, independent='test_replay')

## 1. Properties

- A property is a rule that holds for every valid input: sorting twice equals sorting once; decode(encode(x)) == x.
- Check it over many generated inputs (`random.Random(seed)` for repeatability); Hypothesis automates this.
- Good properties: round-trips, invariants (length kept), idempotence, comparison with a simple reference version.

In [ ]:
import random, json
rng = random.Random(0)
for _ in range(200):
    xs = [rng.randint(-50, 50) for _ in range(rng.randint(0, 10))]
    assert sorted(sorted(xs)) == sorted(xs)
    assert len(sorted(xs)) == len(xs)
    assert json.loads(json.dumps(xs)) == xs
print("200 cases passed")

**Exercise 1.** Round-trip property: does `json.loads(json.dumps(x)) == x` hold for `x = {"a": [1, 2]}`?

In [ ]:
predicted = ___
check(1, predicted)

**Exercise 2.** Idempotence: what is `" hi ".strip().strip()` compared with one `strip()`? Write `True` if equal.

In [ ]:
predicted = ___
check(2, predicted)

**Exercise 3.** Property: `float(str(x)) == x` for these floats. Does it hold for all of them?

In [ ]:
values = [0.1, 2.5, -3.75, 1e-9]
holds = all(float(str(x)) == x for x in values)
predicted = ___
check(3, predicted)

## 2. Table-driven tests

- List cases as `(input, expected)` rows and loop over them; adding a case is one line.
- Include boundaries: empty, one item, the limit, just past the limit, invalid input.
- Use `subTest` (unittest) or `pytest.mark.parametrize` so each row reports separately.

In [ ]:
def grade(score):
    if not 0 <= score <= 100:
        raise ValueError(score)
    return "pass" if score >= 50 else "fail"

cases = [(0, "fail"), (49, "fail"), (50, "pass"), (100, "pass")]
print(all(grade(s) == want for s, want in cases))

**Exercise 4.** Which boundary score is the first `"pass"`?

In [ ]:
def grade(score):
    return "pass" if score >= 50 else "fail"

results = [grade(s) for s in [49, 50, 51]]
first_pass = [49, 50, 51][results.index(___)]
check(4, first_pass)

**Exercise 5.** A test dashboard receives (observed, expected) pairs. Count every failing observation, including repeated identical failures. The input order has no effect on the count. Repair `mismatches` and write `regression(candidate)`. Your test must pass for the repair and raise AssertionError for the original defect.

Earlier practice, for discussion:

Count the failing rows of the table.

```python
def double(x):
    return x + x

cases = [(1, 2), (0, 0), (3, 9), (-1, -2)]
failures = [x for x, want in cases if double(x) != want]
predicted = ___
```

In [ ]:
def mismatches(rows):
    return sum(actual != expected for actual, expected in dict(rows).items())

def regression(candidate):
    raise NotImplementedError


def observe(repair):
    def original(rows):
        return sum(actual != expected for actual, expected in dict(rows).items())

    regression(repair)
    try:
        regression(original)
    except AssertionError:
        detected = True
    else:
        detected = False
    inputs = [([(1, 2), (1, 2), (3, 3)],), ([],), ([(2, 2), (2, 3)],)]
    return [repair(*args) for args in inputs], detected

check(5, mismatches, observe=observe)

## 3. Integration tests

- Integration tests run real pieces together: a real SQLite database in `:memory:` or a temp folder.
- Use `tempfile.TemporaryDirectory()` so files are cleaned up.
- Mock only true external boundaries (network, clock, payments); test your own code for real.

In [ ]:
import sqlite3, tempfile, os
with tempfile.TemporaryDirectory() as folder:
    path = os.path.join(folder, "app.db")
    with sqlite3.connect(path) as conn:
        conn.execute("CREATE TABLE t (x)")
        conn.execute("INSERT INTO t VALUES (1)")
    print(sqlite3.connect(path).execute("SELECT COUNT(*) FROM t").fetchone())
print(os.path.exists(folder))

**Exercise 6.** After the `TemporaryDirectory` block, does the folder still exist?

In [ ]:
predicted = ___
check(6, predicted)

**Exercise 7.** Which of these should be mocked: `"database"`, `"payment API"`, `"your parser"`?

In [ ]:
answer = ___
check(7, answer)

### Catch a stateful regression

Implement two interchangeable delivery stores, then write an integration test for first delivery, replay and interleaved keys. Introduce a deliberate last-key-only bug and demonstrate that your test fails. Explain what additional fixture is needed to test persistence across restarts.

Read the ticket here and identify a failure case. After the independent build, use the engineering submission cell to implement it and run your regression tests. This ticket is reviewed from the submitted code and tests.

## Independent build

Write test_replay(deliver) for an offline Order Desk delivery service. deliver(key) returns True the first time each key is accepted and False on repeats. Include an intervening different key. Your test should complete for a conforming service and raise AssertionError for a service that remembers only the most recent key.

In a short review, explain which observable invariant the faulty service violates and why consecutive duplicates alone miss it.

In [ ]:
# Write your implementation here.

In [ ]:
def acceptance_cases(submission):
    def delivery_double(history):

        def deliver(key):
            fresh = key not in history
            history.append(key)
            return fresh
        return deliver
    from collections import deque
    submission(delivery_double([]))
    try:
        submission(delivery_double(deque(maxlen=1)))
    except AssertionError:
        pass
    else:
        raise AssertionError('A nonconsecutive duplicate escaped detection')

check.assess('test_replay', acceptance_cases)

### Engineering submission

Catch a stateful regression. Include your code, test results and review.

In [ ]:
# Implement the ticket and run your tests here, or submit the project files.

## Quick quiz

Replace each `'?'` in the cell below with your choice and run it. The answers are hidden, but each one is checked.

**1.** A property-based test checks…

   - **a)** An invariant across many generated inputs
   - **b)** Only performance
   - **c)** One example

**2.** Why seed random test data?

   - **a)** Faster tests
   - **b)** Failures become reproducible
   - **c)** More coverage

**3.** Integration tests against a temporary SQLite DB check…

   - **a)** Only pure functions
   - **b)** The network
   - **c)** Your code with a real adapter

In [ ]:
from quiz import Quiz

quiz = Quiz('advanced/14', {1: 'feb2d351bd74', 2: 'b5ed09fcd6ed', 3: 'e36e6d620f68'})
quiz.answer(1, '?')
quiz.answer(2, '?')
quiz.answer(3, '?')

**Next:** [15 — Observability, Structured Logging, Metrics, and SLOs](15_observability_and_slos.ipynb)